# voiceai — Echter Mini-Trainingslauf (1 Zelle, 1 Play, uv)

**3 Schritte:**
1. Oben: **Runtime → Change runtime type → T4 GPU** → Save
2. In der Zelle unten deinen Hugging-Face-Token einfügen (https://huggingface.co/settings/tokens)
3. **Play** drücken. Fertig.

Nutzt **uv** statt pip. Lädt 64 echte LibriSpeech-Clips, Mimi-encodet sie,
trainiert den Stage-1-Adapter auf dem echten Qwen3.5-0.8B. Der Loss sinkt wirklich.

In [ ]:
import os
os.environ['HF_TOKEN'] = 'PASTE_YOUR_HF_TOKEN_HERE'  # <-- nur das hier aendern
assert os.environ['HF_TOKEN'].startswith('hf_'), 'HF-Token noch nicht eingetragen!'

import torch
assert torch.cuda.is_available(), 'Keine GPU! Runtime -> Change runtime type -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

# uv holen (per curl, kein pip)
!curl -LsSf https://astral.sh/uv/install.sh | sh

![ -d voiceai ] || git clone -q https://github.com/chukfinley/voiceai.git
%cd /content/voiceai
!git pull -q

# install + cleanup mit uv (--system = Colabs Python, damit der Kernel es sieht)
!~/.local/bin/uv pip install --system -q -e '.[train,dev]'
!~/.local/bin/uv pip uninstall --system torchvision torchao

# echter Mini-Trainingslauf
!python scripts/colab_realtest.py

Am Ende sollte stehen: `[done] adapters -> runs/realtest/` und der Loss in der
Fortschrittsleiste sollte über die 200 Schritte gefallen sein.

Zu wenig GPU-Speicher? Letzte Zeile ersetzen durch:
`!python scripts/colab_realtest.py --backbone Qwen/Qwen3-0.6B --steps 150`